## Overview

This dataset contains daily stock price data. Preprocessing focuses on financial feature engineering, handling temporal ordering, and preparing the data for time series forecasting.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler


df = pd.read_csv("C:/Users/ADMIN/Downloads/timeserries_models_analysis/data/raw/AAPL.csv")

print(df.shape)
df.info()
df.describe()

(10409, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10409 entries, 0 to 10408
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       10409 non-null  object 
 1   Open       10409 non-null  float64
 2   High       10409 non-null  float64
 3   Low        10409 non-null  float64
 4   Close      10409 non-null  float64
 5   Adj Close  10409 non-null  float64
 6   Volume     10409 non-null  int64  
dtypes: float64(5), int64(1), object(1)
memory usage: 569.4+ KB


,Open,High,Low,Close,Adj Close,Volume
count,10409.000000,10409.000000,10409.000000,10409.000000,10409.000000,1.040900e+04
mean,13.959910,14.111936,13.809163,13.966757,13.350337,3.321778e+08
std,30.169244,30.514878,29.835055,30.191696,29.911132,3.393344e+08
min,0.049665,0.049665,0.049107,0.049107,0.038384,0.000000e+00
25%,0.281964,0.287946,0.274554,0.281250,0.234799,1.247604e+08
50%,0.468750,0.477679,0.459821,0.468750,0.386853,2.199680e+08
75%,14.217857,14.364286,14.043571,14.206071,12.188149,4.126108e+08
max,182.630005,182.940002,179.119995,182.009995,181.778397,7.421641e+09


In [2]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df.set_index('Date', inplace=True)

df.head(10)

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1980-12-12,0.128348,0.128906,0.128348,0.128348,0.100323,469033600
1980-12-15,0.122210,0.122210,0.121652,0.121652,0.095089,175884800
1980-12-16,0.113281,0.113281,0.112723,0.112723,0.088110,105728000
1980-12-17,0.115513,0.116071,0.115513,0.115513,0.090291,86441600
1980-12-18,0.118862,0.119420,0.118862,0.118862,0.092908,73449600
1980-12-19,0.126116,0.126674,0.126116,0.126116,0.098578,48630400
1980-12-22,0.132254,0.132813,0.132254,0.132254,0.103376,37363200
1980-12-23,0.137835,0.138393,0.137835,0.137835,0.107739,46950400
1980-12-24,0.145089,0.145647,0.145089,0.145089,0.113409,48003200


In [3]:
df = df.ffill()
df = df.dropna()

df['price_range'] = df['High'] - df['Low']
df['price_change'] = df['Close'] - df['Open']
df['return'] = df['Close'].pct_change()

df['ma_7'] = df['Close'].rolling(window=7).mean()
df['ma_30'] = df['Close'].rolling(window=30).mean()
df['volatility'] = df['Close'].rolling(window=7).std()
df = df.dropna() # Drop NaN from rolling

df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 10380 entries, 1981-01-26 to 2022-03-24
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open          10380 non-null  float64
 1   High          10380 non-null  float64
 2   Low           10380 non-null  float64
 3   Close         10380 non-null  float64
 4   Adj Close     10380 non-null  float64
 5   Volume        10380 non-null  int64  
 6   price_range   10380 non-null  float64
 7   price_change  10380 non-null  float64
 8   return        10380 non-null  float64
 9   ma_7          10380 non-null  float64
 10  ma_30         10380 non-null  float64
 11  volatility    10380 non-null  float64
dtypes: float64(11), int64(1)
memory usage: 1.0 MB


### Datetime Processing

The date column was converted to datetime format and set as the index to ensure correct chronological ordering of observations.

### Temporal Characteristics

Unlike high-frequency data, stock data naturally contains gaps due to non-trading days (e.g., weekends and holidays). These gaps were preserved as they reflect real-world market behaviour.

### Missing Value Handling

The dataset contained minimal missing values. Any missing entries were handled using forward filling to maintain continuity without introducing unrealistic values.

### Feature Engineering

Several domain-specific features were created to capture market behaviour:

- Price range (High − Low) to represent daily volatility
- Price change (Close − Open) to capture intraday movement
- Returns (percentage change in closing price) to model relative price movement

### Rolling Statistics

Rolling window features were introduced to capture trends and volatility:

- Moving averages (7-day and 30-day) for trend smoothing
- Rolling standard deviation to represent short-term volatility

These operations introduce missing values at the beginning of the dataset due to insufficient historical data, which were removed.

In [4]:
scaler = StandardScaler()

df_scaled = pd.DataFrame(
    scaler.fit_transform(df),
    index=df.index,
    columns=df.columns
)

train_size = int(len(df_scaled) * 0.8)
train = df_scaled[:train_size]
test = df_scaled[train_size:]

def create_sequences(data, window=30):
    X, y = [], []
    
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(data[i+window])
    
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train.values, window=30)
X_test, y_test = create_sequences(test.values, window=30)


### Scaling

All features were standardised to ensure consistent scale across variables, improving model performance and training stability.

### Sequence Construction

A sliding window approach was used to convert the time series into supervised learning samples. Each input sequence consists of 30 time steps (approximately one month of trading days), used to predict the next observation.

## Summary
- Converted date column to datetime index
- Preserved natural temporal gaps in trading data
- Engineered financial features (returns, price range, etc.)
- Applied rolling statistics to capture trends and volatility
- Removed initial rows with insufficient history
- Normalised features for modelling
- Generated sequences for forecasting

In [5]:
df_scaled.to_csv("C:/Users/ADMIN/Downloads/timeserries_models_analysis/data/cleaned/apple_cleaned.csv")